# Notebook 18g: Variance Estimation from Permutations

## Purpose
Validate the trained Degree Signature NN model (from notebook 18f) on 20
degree-preserving permuted networks. This establishes that the model captures
degree-driven pathway formation and estimates variance for statistical anomaly
detection in notebook 18h.

## Method
For each of 20 permutations:
1. Load permuted edges (degree-preserved, topology randomized)
2. Compute pathway counts in permutation
3. Extract degree signatures (same binning strategy as training)
4. Predict using trained model
5. Compare predictions to actual counts

If model is good:
- Predictions should match permutation counts (r ≈ 0.80-0.90)
- Because permutations preserve degree but randomize topology
- This validates model as "degree-conditioned null"

## Key Insight
The variance across permutations represents uncertainty due to topology randomness.
This enables statistical significance testing in notebook 18h (anomaly detection).

## Inputs
- `results/pathway_nn/trained_models/{metapath}_Degree_Sig_NN.pt`
  - Trained model from notebook 18f
- `data/permutations/{001-020}.hetmat/edges/{edge_type}.sparse.npz`
  - 20 degree-preserving permuted networks

## Outputs
- `results/pathway_nn/variance_analysis/{metapath}_variance_estimates.csv`
  - Mean, std, percentiles for each degree bin combination
- `results/pathway_nn/variance_analysis/{metapath}_permutation_metrics.csv`
  - Correlation, MAE, RMSE for each permutation
- `results/pathway_nn/variance_analysis/{metapath}_validation_summary.json`
  - Aggregate statistics (mean correlation, etc.)
- `results/pathway_nn/variance_analysis/permutation_{id}_predictions.npy`
  - Individual permutation predictions (20 files)
- `results/pathway_nn/variance_analysis/all_permutations_results.npz`
  - Aggregated results (compressed)
- `results/pathway_nn/variance_analysis/{metapath}_permutation_validation.png`
  - Validation plots (correlation, MAE, scatter, variance dist)

## Usage
```bash
# Local execution
jupyter nbconvert --execute notebooks/18g_variance_estimation.ipynb

# HPC execution with papermill
papermill notebooks/18g_variance_estimation.ipynb \
    notebooks/executed/18g_variance_estimation_executed.ipynb \
    -p metapath "CbGpPW"
```

## References
- Himmelstein et al. (2017). Systematic integration of biomedical knowledge
  prioritizes drugs for repurposing. eLife. https://doi.org/10.7554/eLife.26726
  - XSwap algorithm for degree-preserving permutations

In [ ]:
# Papermill parameters
metapath = 'CbGpPW'
edge1_type = 'CbG'
edge2_type = 'GpPW'
n_permutations = 20
first_perm_id = 1
n_degree_bins = 10
n_inter_bins = 10
random_seed = 42

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import json
from pathlib import Path
from scipy.stats import pearsonr, spearmanr
import sys

repo_dir = Path.cwd().parent
sys.path.insert(0, str(repo_dir))

from src.models.degree_signature_nn import DegreeSignatureNN
from src.intermediate_signatures import (
    compute_intermediate_signature,
    create_degree_bins,
    assign_to_bins,
    extract_training_features
)

print(f"Variance estimation for {metapath}")
print(f"Validating model on {n_permutations} permutations")

In [ ]:
# Load trained model from notebook 18f
model_file = (repo_dir / 'results' / 'pathway_nn' / 'trained_models' /
              f'{metapath}_Degree_Sig_NN.pt')
if not model_file.exists():
    raise FileNotFoundError(
        f"Trained model not found: {model_file}\n"
        "Please run notebook 18f_train_degree_signature_nn.ipynb first!"
    )

model = DegreeSignatureNN.load(model_file)
print(f"✓ Loaded trained model: {model_file}")
print(f"  Model: {model}")

In [ ]:
# Validate model on each permutation
permutation_results = []

for perm_id in range(1, n_permutations + 1):
    print(f"\nProcessing permutation {perm_id:03d}/{n_permutations}...")
    
    # Load permuted edges
    perm_dir = (repo_dir / 'data' / 'permutations' /
                f'{perm_id:03d}.hetmat' / 'edges')
    edge1_file = perm_dir / f'{edge1_type}.sparse.npz'
    edge2_file = perm_dir / f'{edge2_type}.sparse.npz'
    
    if not edge1_file.exists() or not edge2_file.exists():
        print(f"  Warning: Files not found, skipping permutation {perm_id}")
        continue
    
    edge1_matrix = sp.load_npz(str(edge1_file))
    edge2_matrix = sp.load_npz(str(edge2_file))
    
    # Compute pathway counts in this permutation
    pathway_matrix = edge1_matrix @ edge2_matrix
    if pathway_matrix.dtype == bool or pathway_matrix.dtype == np.bool_:
        pathway_matrix = pathway_matrix.astype(np.int32)
    
    # Compute degrees in this permutation
    source_degrees = np.asarray(edge1_matrix.sum(axis=1)).ravel()
    target_degrees = np.asarray(edge2_matrix.sum(axis=1)).ravel()
    
    # Create degree bins (same strategy as 18a)
    source_bins = create_degree_bins(source_degrees, n_degree_bins)
    target_bins = create_degree_bins(target_degrees, n_degree_bins)
    
    # Compute intermediate signatures
    signatures = compute_intermediate_signature(
        edge1_matrix=edge1_matrix,
        edge2_matrix=edge2_matrix,
        source_degrees=source_degrees,
        target_degrees=target_degrees,
        source_bins=source_bins,
        target_bins=target_bins,
        n_intermediate_bins=n_inter_bins
    )
    
    # Extract features
    X_signatures, bin_pairs = extract_training_features(signatures, normalize=True)
    
    # Add source_bin and target_bin as first two features (model expects this)
    source_target_bins = bin_pairs.astype(np.float32)  # (n_bins, 2)
    X_perm = np.hstack([source_target_bins, X_signatures])
    
    # Aggregate pathway counts by bins
    pathway_coo = pathway_matrix.tocoo()
    pathway_dict = {(i, j): v for i, j, v in
                    zip(pathway_coo.row, pathway_coo.col, pathway_coo.data)}
    
    source_bin_assignments = assign_to_bins(source_degrees, source_bins)
    target_bin_assignments = assign_to_bins(target_degrees, target_bins)
    
    bin_pathway_counts = {}
    for (i, j), count in pathway_dict.items():
        src_bin = source_bin_assignments[i]
        tgt_bin = target_bin_assignments[j]
        key = (src_bin, tgt_bin)
        if key not in bin_pathway_counts:
            bin_pathway_counts[key] = []
        bin_pathway_counts[key].append(count)
    
    # Compute actual mean counts for each bin pair
    y_perm_actual = []
    for src_bin, tgt_bin in bin_pairs:
        counts = bin_pathway_counts.get((src_bin, tgt_bin), [0])
        y_perm_actual.append(np.mean(counts))
    y_perm_actual = np.array(y_perm_actual)
    
    # Predict using trained model
    y_perm_predicted = model.predict(X_perm)
    
    # Compute validation metrics
    r, _ = pearsonr(y_perm_predicted, y_perm_actual)
    mae = np.mean(np.abs(y_perm_predicted - y_perm_actual))
    rmse = np.sqrt(np.mean((y_perm_predicted - y_perm_actual)**2))
    
    print(f"  r = {r:.4f}, MAE = {mae:.4f}, RMSE = {rmse:.4f}")
    
    # Store results
    permutation_results.append({
        'perm_id': perm_id,
        'correlation': r,
        'mae': mae,
        'rmse': rmse,
        'predictions': y_perm_predicted,
        'actual': y_perm_actual,
        'bin_pairs': bin_pairs
    })

print(f"\n✓ Processed {len(permutation_results)} permutations")

In [ ]:
# Compute aggregate statistics across permutations
correlations = [r['correlation'] for r in permutation_results]
maes = [r['mae'] for r in permutation_results]
rmses = [r['rmse'] for r in permutation_results]

print(f"\n{'='*70}")
print(f"VALIDATION ACROSS {len(permutation_results)} PERMUTATIONS")
print(f"{'='*70}")
print(f"\nPearson Correlation:")
print(f"  Mean:   {np.mean(correlations):.4f}")
print(f"  Std:    {np.std(correlations):.4f}")
print(f"  Range:  [{np.min(correlations):.4f}, {np.max(correlations):.4f}]")
print(f"\nMean Absolute Error:")
print(f"  Mean:   {np.mean(maes):.4f}")
print(f"  Std:    {np.std(maes):.4f}")
print(f"  Range:  [{np.min(maes):.4f}, {np.max(maes):.4f}]")
print(f"\nRoot Mean Squared Error:")
print(f"  Mean:   {np.mean(rmses):.4f}")
print(f"  Std:    {np.std(rmses):.4f}")
print(f"  Range:  [{np.min(rmses):.4f}, {np.max(rmses):.4f}]")
print(f"{'='*70}")

In [ ]:
# Estimate variance for each unique bin combination
print(f"\nEstimating variance for each bin combination...")

# Collect all unique bin combinations
all_bin_pairs = permutation_results[0]['bin_pairs']
n_bins = len(all_bin_pairs)

# For each bin combination, collect actual counts across all permutations
variance_estimates = []

for bin_idx, (src_bin, tgt_bin) in enumerate(all_bin_pairs):
    # Get actual counts for this bin across all permutations
    counts_across_perms = [
        permutation_results[perm_idx]['actual'][bin_idx]
        for perm_idx in range(len(permutation_results))
    ]
    
    # Compute statistics
    mean_count = np.mean(counts_across_perms)
    std_count = np.std(counts_across_perms)
    median_count = np.median(counts_across_perms)
    q25 = np.percentile(counts_across_perms, 25)
    q75 = np.percentile(counts_across_perms, 75)
    q025 = np.percentile(counts_across_perms, 2.5)
    q975 = np.percentile(counts_across_perms, 97.5)
    
    # Get model prediction (from first permutation, should be similar)
    model_pred = permutation_results[0]['predictions'][bin_idx]
    
    variance_estimates.append({
        'source_bin': src_bin,
        'target_bin': tgt_bin,
        'mean_count_across_perms': mean_count,
        'std_count_across_perms': std_count,
        'median_count': median_count,
        'q25': q25,
        'q75': q75,
        'ci_lower_95': q025,
        'ci_upper_95': q975,
        'model_prediction': model_pred,
        'n_permutations': len(permutation_results)
    })

variance_df = pd.DataFrame(variance_estimates)
print(f"✓ Computed variance estimates for {len(variance_df)} bin combinations")
print(f"\nVariance Statistics:")
mean_std = variance_df['std_count_across_perms'].mean()
print(f"  Mean std across bins: {mean_std:.4f}")
print(f"  Median std:           {variance_df['std_count_across_perms'].median():.4f}")
print(f"  Max std:              {variance_df['std_count_across_perms'].max():.4f}")

In [ ]:
# Create validation visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Correlation across permutations
axes[0, 0].bar(range(1, len(permutation_results) + 1), correlations,
               color='steelblue', alpha=0.7)
axes[0, 0].axhline(np.mean(correlations), color='red', linestyle='--',
                   label=f'Mean = {np.mean(correlations):.4f}')
axes[0, 0].set_xlabel('Permutation ID', fontsize=11)
axes[0, 0].set_ylabel('Pearson Correlation', fontsize=11)
axes[0, 0].set_title('Model Performance Across Permutations', fontsize=12,
                     fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. MAE across permutations
axes[0, 1].bar(range(1, len(permutation_results) + 1), maes,
               color='coral', alpha=0.7)
axes[0, 1].axhline(np.mean(maes), color='red', linestyle='--',
                   label=f'Mean = {np.mean(maes):.4f}')
axes[0, 1].set_xlabel('Permutation ID', fontsize=11)
axes[0, 1].set_ylabel('Mean Absolute Error', fontsize=11)
axes[0, 1].set_title('MAE Across Permutations', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Example scatter plot from first permutation
perm_0 = permutation_results[0]
axes[1, 0].scatter(perm_0['actual'], perm_0['predictions'], alpha=0.6, s=30)
axes[1, 0].plot([perm_0['actual'].min(), perm_0['actual'].max()],
                [perm_0['actual'].min(), perm_0['actual'].max()],
                'r--', lw=2, label='Perfect prediction')
axes[1, 0].set_xlabel('Actual Count (Permutation 1)', fontsize=11)
axes[1, 0].set_ylabel('Predicted Count', fontsize=11)
axes[1, 0].set_title(f'Permutation 1: r = {perm_0["correlation"]:.4f}',
                     fontsize=12, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Variance distribution
axes[1, 1].hist(variance_df['std_count_across_perms'], bins=30,
                color='mediumpurple', alpha=0.7, edgecolor='black')
axes[1, 1].axvline(mean_std, color='red', linestyle='--', lw=2,
                   label=f'Mean = {mean_std:.2f}')
axes[1, 1].set_xlabel('Std Dev of Counts Across Permutations', fontsize=11)
axes[1, 1].set_ylabel('Frequency', fontsize=11)
axes[1, 1].set_title('Distribution of Variance Estimates', fontsize=12,
                     fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()

# Save plot
output_dir = repo_dir / 'results' / 'pathway_nn' / 'variance_analysis'
output_dir.mkdir(parents=True, exist_ok=True)
plot_file = output_dir / f'{metapath}_permutation_validation.png'
plt.savefig(plot_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved validation plot: {plot_file}")

In [ ]:
# Save per-permutation predictions
print(f"\nSaving per-permutation predictions...")
for perm_result in permutation_results:
    perm_id = perm_result['perm_id']
    pred_file = output_dir / f'permutation_{perm_id:03d}_predictions.npy'
    np.save(pred_file, perm_result['predictions'])

print(f"✓ Saved {len(permutation_results)} prediction files")

# Save aggregated results (compressed)
aggregated_file = output_dir / 'all_permutations_results.npz'
np.savez_compressed(
    aggregated_file,
    correlations=np.array(correlations),
    maes=np.array(maes),
    rmses=np.array(rmses),
    perm_ids=np.array([r['perm_id'] for r in permutation_results])
)
print(f"✓ Saved aggregated results: {aggregated_file}")

In [ ]:
# Save variance estimates
variance_file = output_dir / f'{metapath}_variance_estimates.csv'
variance_df.to_csv(variance_file, index=False)
print(f"✓ Saved variance estimates: {variance_file}")

# Save permutation metrics
metrics_df = pd.DataFrame({
    'permutation_id': [r['perm_id'] for r in permutation_results],
    'correlation': correlations,
    'mae': maes,
    'rmse': rmses
})
metrics_file = output_dir / f'{metapath}_permutation_metrics.csv'
metrics_df.to_csv(metrics_file, index=False)
print(f"✓ Saved permutation metrics: {metrics_file}")

# Save summary statistics
summary = {
    'metapath': metapath,
    'n_permutations': len(permutation_results),
    'mean_correlation': float(np.mean(correlations)),
    'std_correlation': float(np.std(correlations)),
    'mean_mae': float(np.mean(maes)),
    'std_mae': float(np.std(maes)),
    'mean_rmse': float(np.mean(rmses)),
    'std_rmse': float(np.std(rmses)),
    'n_bin_combinations': len(variance_df),
    'mean_variance': float(variance_df['std_count_across_perms'].mean())
}

summary_file = output_dir / f'{metapath}_validation_summary.json'
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✓ Saved summary: {summary_file}")

print(f"\n{'='*70}")
print(f"VARIANCE ESTIMATION COMPLETE: {metapath}")
print(f"{'='*70}")
print(f"\nKey Results:")
print(f"  Validated on {len(permutation_results)} permutations")
mean_corr = np.mean(correlations)
std_corr = np.std(correlations)
print(f"  Mean correlation: {mean_corr:.4f} ± {std_corr:.4f}")
print(f"  Variance estimates for {len(variance_df)} degree bin combinations")
print(f"  Results saved to: {output_dir}")
print(f"\n{'='*70}")
print(f"NEXT STEP: Run notebook 18h_anomaly_detection.ipynb")
print(f"  → Use variance estimates to compute z-scores")
print(f"  → Compare anomaly scores to DWPC")
print(f"  → Identify novel biological associations")
print(f"{'='*70}")